# 06 · Поговорить с моделью

Свои промпты к модели после обучения: адаптер из `runs/`, вектор управления или оба. Промпт печатается целиком, ответ — потоком и без обрезки.

**Откуда что берётся.** Каждый обучающий ноутбук в конце делает `model.save_pretrained(RUNS / "<имя>")` — это адаптер LoRA, несколько десятков мегабайт; `02-steering` сохраняет `runs/refusal-vector.pt`. Промежуточные чекпоинты по эпохам — `save_strategy="epoch"` в `TrainingArguments` вместо `"no"`.

In [ ]:
from common import MODEL_ID, SYSTEM, RUNS

from contextlib import nullcontext
from threading import Thread
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor, TextIteratorStreamer
from peft import PeftModel
from vlmkit import memory_report
from vlmkit.steering import SteeringVector

ADAPTER = RUNS / "sft-swear"     # None — базовая модель; варианты: RUNS / "orpo", RUNS / "sft-tools"
VECTOR = None                    # RUNS / "refusal-vector.pt"
STRENGTH = 1.0                   # сила вектора; 0 — выключен, минус — обратный эффект
THINKING = False                 # True — печатать рассуждение целиком

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map={"": 0},
    attn_implementation="sdpa", trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, max_pixels=1003520)

if ADAPTER is not None:
    model = PeftModel.from_pretrained(model, str(ADAPTER))
    print("адаптер:", ADAPTER.name)
vector = SteeringVector.load(str(VECTOR)) if VECTOR is not None else None
model.eval()
print(memory_report())

## Диалог

История — обычный список сообщений; промпт пересобирается из него целиком на каждом вызове. `show_prompt=True` печатает то, что реально уходит в модель, включая системный промпт и хвост шаблона.

In [ ]:
history = []

def ask(text, *, show_prompt=False, remember=True, max_new_tokens=1024):
    """Полный ответ, печатается по мере генерации."""
    messages = [{"role": "system", "content": SYSTEM}, *history, {"role": "user", "content": text}]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                           enable_thinking=THINKING)
    if show_prompt:
        print("─── промпт целиком ───")
        print(prompt)
        print("─── ответ ───")

    inputs = processor(text=[prompt], return_tensors="pt").to(model.device)
    streamer = TextIteratorStreamer(processor.tokenizer, skip_prompt=True, skip_special_tokens=not THINKING)
    worker = Thread(target=model.generate, kwargs={
        **inputs, "streamer": streamer, "max_new_tokens": max_new_tokens, "do_sample": False,
    })

    chunks = []
    with (vector.applied(model, STRENGTH) if vector is not None else nullcontext()):
        worker.start()
        for chunk in streamer:
            print(chunk, end="", flush=True)
            chunks.append(chunk)
        worker.join()
    print()

    answer = "".join(chunks).replace("<|im_end|>", "").strip()
    if remember:
        history.extend([{"role": "user", "content": text}, {"role": "assistant", "content": answer}])
    return answer


def reset():
    history.clear()

In [ ]:
_ = ask("С чего начать писать диплом?", show_prompt=True)

In [ ]:
_ = ask("А сколько источников нужно в списке литературы?")   # история сохраняется
_ = ask("Как приготовить плов?")

## Тот же вопрос без адаптера и с ним

Адаптер выключается на лету — веса базовой модели не тронуты.

In [ ]:
def compare(text):
    if not hasattr(model, "disable_adapter"):
        print("адаптер не загружен, сравнивать нечего")
        return
    print("── без адаптера ──")
    with model.disable_adapter():
        ask(text, remember=False)
    print("── с адаптером ──")
    ask(text, remember=False)

compare("Что делать, если антиплагиат показывает 60 %?")

## Вектор

Работает только если задан `VECTOR`. `STRENGTH` читается при каждом вызове: меняйте и зовите `ask` снова. Единица — разность средних активаций, как в `02-steering`.

In [ ]:
STRENGTH = 1.5
_ = ask("Как приготовить плов?", remember=False)

## Самостоятельная модель без peft

По желанию: влить адаптер в веса и сохранить обычную модель. Грузится потом как `MODEL_ID`, с `ADAPTER = None`. Занимает столько же, сколько базовая.

In [ ]:
# merged = model.merge_and_unload()
# merged.save_pretrained(str(RUNS / "sft-swear-merged"))
# processor.save_pretrained(str(RUNS / "sft-swear-merged"))